# Trustworthy AI in practice: explaining and auditing a heart-disease classifier

**Dataset:** UCI Heart Disease - Cleveland Clinic Foundation subset (303 patients, 1981–1984)
**Tools:** scikit-learn · [SHAP](https://shap.readthedocs.io) · [Fairlearn](https://fairlearn.org)

A model that is *accurate on average* is not automatically a model we can *trust*. In this notebook we train a heart-disease classifier and then ask the two questions any clinician, patient or regulator would ask:

1. **Why does the model say what it says?** → *Explainable AI (XAI)* with SHAP, both **globally** (how the model behaves overall) and **locally** (why *this* patient got *this* prediction).
2. **Does the model work equally well for everyone?** → *Fairness* with Fairlearn, auditing the model across **sex** and **age groups**.

### Structure
| Part | Topic |
|---|---|
| 0 | Setup |
| 1 | The data |
| 2 | A baseline model |
| 3 | Explainability with SHAP (global → local → pitfalls) |
| 4 | Fairness with Fairlearn (sex, age) |

## 0. Setup

If a package is missing, uncomment and run the `%pip` line (restart the kernel afterwards).

In [ ]:
# %pip install -q scikit-learn pandas matplotlib shap fairlearn

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             recall_score, precision_score, confusion_matrix, ConfusionMatrixDisplay)

import shap
import fairlearn
from fairlearn.metrics import (MetricFrame, count, selection_rate, true_positive_rate,
                               false_positive_rate, false_negative_rate,
                               equalized_odds_difference, demographic_parity_difference)
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)

print(f"scikit-learn {sklearn.__version__} | shap {shap.__version__} | fairlearn {fairlearn.__version__}")

## 1. The data - and who is (not) in it

Each row is a patient referred for **coronary angiography** at the Cleveland Clinic between 1981 and 1984. The label (`num`) is the angiographic result: 0 = less than 50% narrowing of any major vessel, 1–4 = increasing disease. As is standard, we turn it into a binary target: **disease present (num > 0) vs absent**.

Before training anything, a trustworthy-AI mindset asks:
- **Who collected the data, when, and on whom?** One US hospital, 40+ years ago, patients already suspected of heart disease.
- **Who is under-represented?** As we'll see, about two thirds of the patients are men.

The loader below tries the official UCI file first and falls back to a public mirror (the [PMLB](https://github.com/EpistasisLab/pmlb) copy of the same file, re-coded back to the UCI format). A local copy is cached in `cleveland.csv`.

In [ ]:
LOCAL_FILE = Path("data/processed.cleveland.data")

RAW_COLS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]

raw = pd.read_csv(LOCAL_FILE, header=None, names=RAW_COLS, na_values="?")
print(f"Loaded {raw.shape[0]} rows x {raw.shape[1]} columns")
raw.head()

### Data dictionary

| Raw | New detailed name | Meaning |
|---|---|---|
| age | `age` | age in years |
| sex | `sex` | 1 = male, 0 = female (as recorded in the source) |
| cp | `chest_pain` | typical angina / atypical angina / non-anginal pain / asymptomatic |
| trestbps | `resting_bp` | resting blood pressure (mm Hg) |
| chol | `cholesterol` | serum cholesterol (mg/dl) |
| fbs | `fasting_bs_high` | fasting blood sugar > 120 mg/dl |
| restecg | `resting_ecg` | normal / ST-T abnormality / left-ventricular hypertrophy |
| thalach | `max_hr` | maximum heart rate reached in the exercise test |
| exang | `exercise_angina` | exercise-induced angina (1 = yes) |
| oldpeak | `st_depression` | ST depression induced by exercise relative to rest |
| slope | `st_slope` | slope of peak-exercise ST segment: upsloping / flat / downsloping |
| ca | `n_vessels` | number of major vessels (0–3) coloured by fluoroscopy |
| thal | `thal` | thallium stress test: normal / fixed defect / reversible defect |
| num | `target` | **1 = disease present** (num > 0), 0 = absent |

We drop the 6 rows with missing values (in `ca` and `thal`) and give the categorical codes readable names so the explanations later are easier to read.

In [ ]:
df = (raw.dropna()
         .rename(columns={"cp": "chest_pain", "trestbps": "resting_bp", "chol": "cholesterol",
                          "fbs": "fasting_bs_high", "restecg": "resting_ecg", "thalach": "max_hr",
                          "exang": "exercise_angina", "oldpeak": "st_depression", "slope": "st_slope",
                          "ca": "n_vessels"})
         .reset_index(drop=True))

df["target"] = (df.pop("num") > 0).astype(int)
df["chest_pain"] = df["chest_pain"].map({1: "typical", 2: "atypical", 3: "non_anginal", 4: "asymptomatic"})
df["resting_ecg"] = df["resting_ecg"].map({0: "normal", 1: "st_t_abnormal", 2: "lv_hypertrophy"})
df["st_slope"] = df["st_slope"].map({1: "up", 2: "flat", 3: "down"})
df["thal"] = df["thal"].map({3: "normal", 6: "fixed_defect", 7: "reversible_defect"})
for c in ["sex", "fasting_bs_high", "exercise_angina", "n_vessels", "age", "resting_bp", "cholesterol", "max_hr"]:
    df[c] = df[c].astype(int)

# Sensitive attributes used in the fairness audit (kept as readable labels)
sex_label = df["sex"].map({0: "female", 1: "male"}).rename("sex")
age_group = pd.cut(df["age"], bins=[0, 49, 59, 120], labels=["<50", "50-59", "60+"]).astype(str).rename("age_group")

print(df.shape)
df.head()

### Who is in the data?

In [ ]:
AGE_ORDER = ["<50", "50-59", "60+"]

def group_summary(groups, order=None):
    out = (pd.DataFrame({"group": groups, "target": df["target"]})
              .groupby("group", observed=True)["target"]
              .agg(n="size", prevalence="mean")
              .assign(share=lambda t: t["n"] / t["n"].sum())
              .round(3))
    return out if order is None else out.reindex(order)

display(group_summary(sex_label))
display(group_summary(age_group, order=AGE_ORDER))
display(group_summary(sex_label + " / " + age_group,
                      order=[f"{s} / {a}" for s in ["female", "male"] for a in AGE_ORDER]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

sex_label.value_counts().reindex(["female", "male"]).plot.bar(ax=axes[0], color=["#af73cbbe", "#299188"], rot=0)
axes[0].set_title("Patients by sex"); axes[0].set_ylabel("count")

df.groupby(sex_label)["target"].mean().reindex(["female", "male"]).plot.bar(
    ax=axes[1], color=["#af73cbbe", "#299188"], rot=0)
axes[1].set_title("Disease prevalence by sex"); axes[1].set_ylim(0, 1); axes[1].set_xlabel("")

df.groupby(age_group)["target"].mean().reindex(["<50", "50-59", "60+"]).plot.bar(ax=axes[2], color="#c14a4a", rot=0)
axes[2].set_title("Disease prevalence by age group"); axes[2].set_ylim(0, 1); axes[2].set_xlabel("")
plt.tight_layout(); plt.show()

## 2. A baseline model

We one-hot encode the categorical features and train two models:
- **Logistic regression** - an *intrinsically interpretable* baseline (one coefficient per feature).
- **Random forest** - a *black-box* model that we will need post-hoc tools to explain.

We stratify the train/test split on **target × sex** so that both sets contain women with and without disease.

In [ ]:
X = pd.get_dummies(df.drop(columns="target"), columns=["chest_pain", "resting_ecg", "st_slope", "thal"],
                   dtype=int)
y = df["target"]
feature_names = X.columns.tolist()
print(f"{X.shape[1]} features:", feature_names)

strata = y.astype(str) + "_" + sex_label
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=strata, random_state=SEED)
print(f"train: {len(X_train)}  test: {len(X_test)}")

In [ ]:
# Logistic regression: scale the features, then fit
lr_scaler = StandardScaler()
X_train_scaled = lr_scaler.fit_transform(X_train)
X_test_scaled = lr_scaler.transform(X_test)

lr = LogisticRegression(C=0.5, max_iter=2000)
lr.fit(X_train_scaled, y_train)

# Random forest: no scaling needed, fit directly
rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=3, max_features="sqrt",
                            random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)

# --- Evaluate logistic regression on the test set ---
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]
lr_pred = (lr_proba >= 0.5).astype(int)

lr_accuracy = accuracy_score(y_test, lr_pred)
lr_balanced_acc = balanced_accuracy_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_proba)
lr_sensitivity = recall_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred)

# --- Evaluate random forest on the test set ---
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_pred = (rf_proba >= 0.5).astype(int)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_balanced_acc = balanced_accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)
rf_sensitivity = recall_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)

# --- Put the two rows of numbers into one table ---
results = pd.DataFrame({
    "Logistic regression": {
        "accuracy": lr_accuracy, "balanced acc.": lr_balanced_acc, "ROC AUC": lr_auc,
        "sensitivity (recall)": lr_sensitivity, "precision": lr_precision,
    },
    "Random forest": {
        "accuracy": rf_accuracy, "balanced acc.": rf_balanced_acc, "ROC AUC": rf_auc,
        "sensitivity (recall)": rf_sensitivity, "precision": rf_precision,
    },
}).T.round(3)

results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))

# Left panel: logistic regression (needs the scaled test set)
ConfusionMatrixDisplay.from_estimator(lr, X_test_scaled, y_test, ax=axes[0], colorbar=False,
                                      display_labels=["no disease", "disease"], cmap="Blues")
axes[0].set_title("Logistic regression")

# Right panel: random forest (uses the raw test set)
ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, ax=axes[1], colorbar=False,
                                      display_labels=["no disease", "disease"], cmap="Blues")
axes[1].set_title("Random forest")

plt.tight_layout(); plt.show()

## 3. Explainability with SHAP

Two types of explanation:
- **Intrinsic**: the model *is* its explanation (e.g., a short decision tree, logistic-regression coefficients).
- **Post-hoc**: we explain an already trained model from the outside (SHAP …).

And two scopes:
- **Global**: what drives the model's predictions *in general*?
- **Local**: why did the model give *this patient* *this* score?

### 3.1 The intrinsically interpretable baseline
For logistic regression on standardised inputs, each coefficient is the change in **log-odds** of disease for a one-standard-deviation increase in that feature.

In [ ]:
coefs = pd.Series(lr.coef_[0], index=feature_names).sort_values()
ax = coefs.plot.barh(figsize=(6, 6), color=np.where(coefs > 0, "#d1495b", "#00798c"))
ax.axvline(0, color="k", lw=0.8)
ax.set_title("Logistic regression: standardised coefficients\n(red → raises predicted risk)")
ax.set_xlabel("change in log-odds per 1 SD"); plt.tight_layout(); plt.show()

### 3.2 Shapley values

Imagine the features of one patient are *players* in a cooperative game, and the "payout" is the model's predicted probability of disease. The **Shapley value** of a feature is its *average marginal contribution* to the payout over all possible orders in which features could join the game. SHAP (SHapley Additive exPlanations; Lundberg & Lee, 2017) computes these for many model types. For tree ensembles, `TreeExplainer` computes them.

The key property is **additivity (local accuracy)**:

$$ f(x) \;=\; \underbrace{\mathbb{E}[f(X)]}_{\text{base value}} \;+\; \sum_{j=1}^{M} \phi_j(x) $$

i.e. *base value + sum of SHAP values = this patient's prediction*.

In [ ]:
explainer = shap.TreeExplainer(rf)
sv_both = explainer(X_test)          # shape: (patients, features, classes)
sv = sv_both[:, :, 1]                # keep the explanation for class 1 = "disease"

Because the random forest outputs probabilities, the SHAP values here are in **probability units**: a SHAP value of +0.10 means "this feature pushed the predicted risk up by 10 percentage points for this patient".

### 3.3 Global explanations

**Bar plot** - mean |SHAP value| per feature: *how much* each feature moves predictions on average (but not in which direction).

In [ ]:
shap.plots.bar(sv, max_display=12)

**Beeswarm plot** - one dot per patient per feature. Horizontal position = SHAP value (effect on predicted risk), colour = the feature's value (red = high, blue = low). This shows *direction* and *spread* as well as magnitude.

In [ ]:
shap.plots.beeswarm(sv, max_display=12)

- Which features push risk **up** when they are high? Which push it **down**?
- Does the model's behaviour agree with clinical intuition (e.g., more vessels with narrowing → higher risk; higher maximum heart rate reached → lower risk)?
- Where does **`sex`** rank? In which direction does being male (red) push the prediction?

### 3.4 Local explanations: one patient at a time

Global plots summarise the model; a clinician sitting with a patient needs a **local** explanation. A **waterfall plot** starts at the base value and adds each feature's SHAP value until it reaches the patient's predicted risk.

We look at two informative cases from the test set:
1. a patient **correctly** flagged as high risk, and
2. a patient **with disease that the model missed** (false negative) 

In [ ]:
pred = rf.predict_proba(X_test)[:, 1]
test_view = X_test.assign(y_true=y_test.values, risk=pred, sex=sex_label.loc[X_test.index].values)

tp_pos = int(np.argmax(np.where(test_view["y_true"] == 1, pred, -1)))       # most confident true positive
fn_mask = (test_view["y_true"] == 1) & (pred < 0.5)
fn_pos = int(np.argmax(np.where(fn_mask, pred, -np.inf))) if fn_mask.any() else None
print("False negatives in the test set:", int(fn_mask.sum()))

cols = ["age", "sex", "chest_pain_asymptomatic", "max_hr", "n_vessels", "thal_normal", "y_true", "risk"]
test_view.iloc[[tp_pos, fn_pos]][cols].round(3)

In [ ]:
print("Case 1: confident true positive")
shap.plots.waterfall(sv[tp_pos], max_display=10)

In [ ]:
print("Case 2: False negative patient")
shap.plots.waterfall(sv[fn_pos], max_display=10)

In [ ]:
def explain_in_words(explanation, row, top_k=3):
    contrib = pd.Series(explanation.values, index=explanation.feature_names)
    risk = explanation.base_values + contrib.sum()
    up = contrib[contrib > 0].sort_values(ascending=False).head(top_k)
    down = contrib[contrib < 0].sort_values().head(top_k)
    fmt = lambda s: ", ".join(f"{f} = {row[f]:g} ({v:+.0%})" for f, v in s.items()) or "none"
    return (f"Predicted risk {risk:.0%}.\n"
            f"  Main factors raising risk:  {fmt(up)}\n"
            f"  Main factors lowering risk: {fmt(down)}")

for name, pos in [("Case 1", tp_pos), ("Case 2", fn_pos)]:
    print(name, "->", explain_in_words(sv[pos], X_test.iloc[pos]), "\n")

 The explanation tells us *why the model believed what it believed*. Would a cardiologist have been fooled by the same information? 

## 4. Fairness with Fairlearn

### 4.0 Which notion of fairness?
Fairness metrics compare a model's behaviour across groups defined by a **sensitive feature** - here **sex** and **age group**. The most common *group* fairness criteria for a binary classifier are:

| Criterion | Requires equal across groups… | Fairlearn |
|---|---|---|
| Demographic parity | selection rate P(ŷ=1) | `demographic_parity_difference` |
| Equal opportunity | true-positive rate (sensitivity) | `true_positive_rate` in a `MetricFrame` |
| Equalized odds | TPR **and** FPR | `equalized_odds_difference` |

In our setting:
- Demographic parity would ask the model to flag women as often as men, even though their recorded prevalence is about half. That is generally **not** what we want in diagnosis.
- **Equal opportunity** asks: *among patients who really have disease, is the model equally likely to catch it regardless of sex or age?* Missing disease (a false negative) is the most serious harm here, so this is our primary lens; equalized odds adds false alarms (FPR).

### 4.1 Getting enough data to measure: out-of-fold predictions
Our 90-patient test set has fewer than 30 women, and only a handful of them have disease: a sensitivity computed on so few patients is almost meaningless. To use **every patient exactly once as a test case**, we use **5-fold cross-validation** and collect the *out-of-fold* predictions: each patient is scored by a model that never saw them during training.

In [ ]:
cv_strata = y.astype(str) + "_" + sex_label
SPLITS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, cv_strata))

oof_proba = cross_val_predict(
    RandomForestClassifier(n_estimators=500, min_samples_leaf=3, max_features="sqrt",
                           random_state=SEED, n_jobs=-1),
    X, y, cv=SPLITS, method="predict_proba"
)[:, 1]
oof_pred = pd.Series((oof_proba >= 0.5).astype(int), index=y.index)

test_women = sex_label.loc[X_test.index] == "female"
print(f"For comparison, the test set has {test_women.sum()} women, "
      f"of whom {y_test[test_women].sum()} have disease.")
print(f"Out-of-fold accuracy: {accuracy_score(y, oof_pred):.3f} | AUC: {roc_auc_score(y, oof_proba):.3f}")

### 4.2 Disaggregated metrics by sex with `MetricFrame`

`MetricFrame` evaluates any metric overall *and* per group, and gives us differences and ratios between groups.

In [ ]:
METRICS = {
    "accuracy": accuracy_score,
    "TPR (sensitivity)": true_positive_rate,
    "FNR (missed disease)": false_negative_rate,
    "FPR (false alarm)": false_positive_rate,
}

mf_sex = MetricFrame(metrics=METRICS, y_true=y, y_pred=oof_pred, sensitive_features=sex_label)
display(mf_sex.overall.to_frame("overall").T.round(3))
display(mf_sex.by_group.round(3))
print("Largest between-group difference per metric:")
display(mf_sex.difference().round(3).to_frame("difference").T)

In [ ]:
PLOT_METRICS = ["TPR (sensitivity)", "FPR (false alarm)"]

def plot_by_group(mf, title, order=None):
    data = mf.by_group if order is None else mf.by_group.reindex(order)
    ax = data[PLOT_METRICS].T.plot.bar(figsize=(8, 3.6), rot=0, width=0.75,
                                              color=["#af73cbbe", "#299188", "#73cbc4bd"][:len(data)])
    ax.set_ylim(0, 1.05); ax.set_title(title); ax.legend(title=mf.by_group.index.name, frameon=False)
    plt.tight_layout(); plt.show()

plot_by_group(mf_sex, "Random forest (out-of-fold): metrics by sex")

### 4.3 By age group

In [ ]:
mf_age = MetricFrame(metrics=METRICS, y_true=y, y_pred=oof_pred, sensitive_features=age_group)
display(mf_age.by_group.reindex(AGE_ORDER).round(3))
plot_by_group(mf_age, "Random forest (out-of-fold): metrics by age group", order=AGE_ORDER)

Note that age is **both** a sensitive attribute *and* a genuine clinical risk factor that the model uses as input. Different error rates across age groups might reflect that the disease presents differently, that some groups are harder to diagnose, or that there is simply less data.

### 4.5 How sure are we? Bootstrap confidence intervals

With groups this small, a 10-point gap in TPR could be noise. We resample patients with replacement 2,000 times and recompute each group's TPR and FPR to get 95% intervals.

In [ ]:

def _rate(y_true, y_pred, positive_class):
    """
    Compute TPR (positive_class=1) or FPR (positive_class=0).
    Assumes binary labels: 0 = control, 1 = cancer.
    """
    mask = y_true == positive_class

    # The rate cannot be computed if no samples belong to this class.
    if mask.sum() == 0:
        return np.nan

    # TPR: proportion predicted as cancer among actual cancer cases.
    # FPR: proportion predicted as cancer among actual controls.
    return y_pred[mask].mean()


def bootstrap_group_rates(y_true, y_pred, groups, n_boot=2000, seed=SEED):
    """
    Calculate TPR and FPR with 95% stratified bootstrap CIs
    for each demographic group.

    Resampling is performed separately for positive and negative
    cases to preserve the original class distribution.
    """
    rng = np.random.default_rng(seed)

    y_true, y_pred, groups = map(
        np.asarray, (y_true, y_pred, groups)
    )

    rows = []

    for g in pd.unique(groups):

        # Select observations belonging to this demographic group.
        mask = groups == g
        yt, yp = y_true[mask], y_pred[mask]

        # Identify positive (cancer) and negative (control) samples.
        idx_pos = np.flatnonzero(yt == 1)
        idx_neg = np.flatnonzero(yt == 0)

        # Calculate observed TPR and FPR.
        estimates = {
            "TPR": _rate(yt, yp, 1),
            "FPR": _rate(yt, yp, 0)
        }

        # Store bootstrap estimates separately for each metric.
        boots = {"TPR": [], "FPR": []}

        for _ in range(n_boot):

            # Sample with replacement within each true class.
            # This preserves the number of positives and negatives.
            sampled_pos = (
                rng.choice(idx_pos, size=len(idx_pos), replace=True)
                if len(idx_pos) > 0 else np.array([], dtype=int)
            )

            sampled_neg = (
                rng.choice(idx_neg, size=len(idx_neg), replace=True)
                if len(idx_neg) > 0 else np.array([], dtype=int)
            )

            # Calculate TPR using resampled cancer cases.
            if len(sampled_pos) > 0:
                boots["TPR"].append(
                    _rate(yt[sampled_pos], yp[sampled_pos], 1)
                )

            # Calculate FPR using resampled controls.
            if len(sampled_neg) > 0:
                boots["FPR"].append(
                    _rate(yt[sampled_neg], yp[sampled_neg], 0)
                )

        # Compute percentile-based 95% bootstrap confidence intervals.
        for metric in ["TPR", "FPR"]:

            if len(boots[metric]) > 0:
                lo, hi = np.percentile(
                    boots[metric], [2.5, 97.5]
                )
            else:
                # Undefined metric when the required class is absent.
                lo, hi = np.nan, np.nan

            rows.append({
                "group": g,
                "metric": metric,
                "estimate": estimates[metric],
                "low": lo,
                "high": hi,
                "n_positive": len(idx_pos),
                "n_negative": len(idx_neg)
            })

    return pd.DataFrame(rows)


def plot_ci(ci, order, title):
    """
    Plot TPR and FPR with 95% bootstrap confidence intervals
    for each demographic group.
    """
    fig, axes = plt.subplots(
        1, 2, figsize=(10, 3.2), sharey=False
    )

    for ax, metric in zip(axes, ["TPR", "FPR"]):

        # Select the metric and arrange groups in the requested order.
        d = ci[ci.metric == metric].set_index("group").loc[order]

        # Exclude groups where the metric or CI is undefined.
        valid = d[["estimate", "low", "high"]].notna().all(axis=1)
        d_valid = d.loc[valid]

        # Plot observed estimates with asymmetric confidence intervals.
        ax.errorbar(
            d_valid.index,
            d_valid["estimate"],
            yerr=[
                d_valid["estimate"] - d_valid["low"],
                d_valid["high"] - d_valid["estimate"]
            ],
            fmt="o",
            capsize=6,
            color="#355c7d",
            ms=8
        )

        ax.set_ylim(0, 1)
        ax.set_title(f"{metric} with 95% bootstrap CI")
        ax.set_ylabel("Rate")
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()



# Compute TPR and FPR with stratified bootstrap 95% CIs
ci_sex = bootstrap_group_rates(
    y, oof_pred, sex_label,
    n_boot=2000,
    seed=SEED
)

ci_age = bootstrap_group_rates(
    y, oof_pred, age_group,
    n_boot=2000,
    seed=SEED
)

# Plot subgroup performance
plot_ci(ci_sex, ["female", "male"], "By sex")
plot_ci(ci_age, AGE_ORDER, "By age group")

# Display results
display(ci_sex.round(3))
display(ci_age.round(3))


What is bootstrapping?
- A statistical technique used to estimate the uncertainty of a result.
- It repeatedly creates new samples from the original dataset by randomly selecting observations with replacement.
- The same observation can appear multiple times in a sample, while others may not appear at all.
How does it work?
1. Randomly resample the original data with replacement.
2. Calculate a metric (e.g., TPR or FPR).
3. Repeat the process 2,000 times.
4. Use the distribution of the results to calculate a 95% confidence interval (CI).
Why is it important?
- A metric calculated on a single dataset is only an estimate.
- Bootstrapping helps us understand how much that estimate might vary.
- It is particularly useful for small datasets, although uncertainty can still be substantial.
- It does not require assuming that the metric follows a normal distribution.
Why are we using it?
- To estimate the uncertainty of TPR and FPR across demographic groups.
- To understand how reliable the subgroup performance estimates are.
- To avoid interpreting small differences between groups as meaningful without considering uncertainty.

## 5. Exercises
1. Pick a female patient from the test set who has disease and is correctly flagged, and one who is a false negative (if any exist; if not, pick a woman with the lowest predicted risk among those with disease). Draw a waterfall plot for each and write 3 sentences: which features drive each prediction, and do they match clinical intuition? Compare with the male false negative in Section 3.4. Are the missed cases missed for similar reasons?
2. Using the beeswarm plot and the mean |SHAP| bar plot, find where sex ranks. Then retrain the random forest without the sex column, recompute out-of-fold predictions, and rerun the MetricFrame by sex. Questions: did the TPR/FPR gaps shrink? Why might removing a sensitive feature not remove disparities (hint: look at chest_pain, thal, max_hr for sex-related patterns)?
3. Repeat the Section 4 out-of-fold audit for the logistic regression pipeline (scaler + LogisticRegression) using the same SPLITS. Compare TPR and FPR by sex and by age group for both models, with bootstrap CIs. Is the more interpretable model fairer?
4. Repeat the Section 4 out-of-fold audit for the logistic regression pipeline (scaler + LogisticRegression) using the same SPLITS. Compare TPR and FPR by sex and by age group for both models, with bootstrap CIs. Is the more interpretable model fairer? Is any difference larger than the uncertainty?
5. The data section showed that female patients under 50 number only 24 (with a prevalence of about 4%). Build a combined sex / age_group variable, run MetricFrame and the bootstrap on the six subgroups, and report n_positive for each. Questions: which subgroups can you not evaluate reliably, and what would you tell a hospital about using this model on them?

### 5.1 First look at mitigation
Mitigation methods try to reduce a gap between groups. Two practical families, both in Fairlearn:
- Post-processing (ThresholdOptimizer): keep the trained model fixed and pick a different decision threshold per group so that a chosen criterion (e.g. equalized odds) is satisfied.
- In-training (ExponentiatedGradient with a constraint such as EqualizedOdds()): retrain while penalizing violation of the constraint.

>Using ThresholdOptimizer(estimator=rf, constraints="true_positive_rate_parity", prefit=True, predict_method="predict_proba") (or "equalized_odds"), fit on X_train with sensitive_features for sex, then predict on X_test. Compute TPR, FPR and accuracy by sex before and after. Questions:
>- Did the gap close? What happened to overall accuracy and to FPR?
>- With so few women in the test set, how much would you trust the result? (Add a bootstrap interval.)